# Data Collection — Pakistan Data Job Postings (Rozee.pk)

**Goal:** Collect Data Analyst / Data Scientist / Data Engineer / Business Analyst 
postings from Rozee.pk to measure real skill demand in the Pakistani market, 
for comparison against global demand (Stack Overflow Survey 2025, Kaggle ai-jobs.net).

**Method:** Scrape via `requests` + `BeautifulSoup`. Site markup was inspected 
manually on July 7, 2026 to confirm selectors before writing the parser.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

SEARCH_SLUGS = [
    "data-analyst-jobs-in-pakistan",
    "data-scientist-jobs-in-pakistan",
    "data-engineer-jobs-in-pakistan",
    "business-analyst-jobs-in-pakistan",
]

BASE_URL = "https://www.rozee.pk/search/{slug}"

def fetch_page(slug, page_num):
    url = BASE_URL.format(slug=slug)
    params = {"page": page_num} if page_num > 1 else {}
    resp = requests.get(url, headers=HEADERS, params=params, timeout=15)
    resp.raise_for_status()
    return resp.text

In [ ]:
SKILL_KEYWORDS = [
    "SQL", "Python", "Power BI", "Excel", "Tableau", "R", "Pandas",
    "Machine Learning", "PostgreSQL", "MySQL", "Data Warehousing",
    "ETL", "Data Modeling", "Statistics", "VBA", "Google Analytics",
    "Looker", "Snowflake", "Spark"
]


In [ ]:

def extract_skills(text):
    found = [kw for kw in SKILL_KEYWORDS if kw.lower() in text.lower()]
    return "; ".join(found)


In [ ]:

def parse_job_cards(html):
    soup = BeautifulSoup(html, "html.parser")
    jobs = []
    cards = soup.select("div.jcont")

    for card in cards:
        title_tag = card.select_one("h3.s-18 a bdi")
        company_tag = card.select_one(".cname bdi a")
        cname_bdi = card.select_one(".cname bdi")
        desc_tag = card.select_one(".jbody bdi")

        title = title_tag.get_text(strip=True) if title_tag else None
        company = company_tag.get_text(strip=True).rstrip(",").strip() if company_tag else None

        location = None
        if cname_bdi:
            full_text = cname_bdi.get_text(" ", strip=True)
            if company:
                location = full_text.replace(company, "").strip(" ,")

        description = desc_tag.get_text(strip=True) if desc_tag else ""

        if title:
            jobs.append({
                "title": title,
                "company": company,
                "location": location,
                "skills": extract_skills(title + " " + description),
                "description_snippet": description,
            })
    return jobs

In [ ]:
html = fetch_page("data-analyst-jobs-in-pakistan", 1)
jobs = parse_job_cards(html)
print(len(jobs))
jobs[:2]

In [8]:
df_raw = pd.read_excel("D:/PK-Data-Job-Analysis/data/raw/pakistan_data_jobs_raw.xlsx")  # adjust path to wherever you uploaded it in Colab
print(df_raw.shape)
print(df_raw.columns.tolist())
df_raw.head()

(43, 5)
['Title', 'Company', 'Location', 'Skill ', 'Category']


,Title,Company,Location,Skill,Category
0,Senior Project Analyst,\nNow IT Solutions,Pakistan,Program ManagementDocumentationData AnalysisAg...,Data Analyst
1,Seniour Analyst,Habib Bank Limited,Karachi,Data AnalysisPerformance ManagementProcess Imp...,Data Analyst
2,Seniour Growth Analyst,Naseeb Enterprise Inc,Lahore,المحاسبة للشركاتData MiningTableauExploratory ...,Data Analyst
3,Market Intelligence,Naseeb Enterprise Inc,Hyderabad,Business WritingResearch and Data AnalysisStor...,Business Intelligence
4,Data and Reporting Analyst,\nMicroMerger (Pvt.) Ltd,Islamabad,ArcGISData AnalyticsPower BI,Reporting Analyst


In [9]:
# Fix the trailing-space column name issue
df_raw.columns = df_raw.columns.str.strip()
print(df_raw.columns.tolist())  # confirm: should show 'Skill' with no space now

# Reuse the same keyword list from the scraper — consistency matters,
# so local and global data get measured against identical skill definitions
SKILL_KEYWORDS = [
    "SQL", "Python", "Power BI", "Excel", "Tableau", "R", "Pandas",
    "Machine Learning", "PostgreSQL", "MySQL", "Data Warehousing",
    "ETL", "Data Modeling", "Statistics", "VBA", "Google Analytics",
    "Looker", "Snowflake", "Spark", "Communication"
]

def extract_skills(text):
    if pd.isna(text):
        return []
    text = str(text).lower()
    return [kw for kw in SKILL_KEYWORDS if kw.lower() in text]

df_raw["skills_matched"] = df_raw["Skill"].apply(extract_skills)
df_raw[["Title", "skills_matched"]].head(10)

['Title', 'Company', 'Location', 'Skill', 'Category']


,Title,skills_matched
0,Senior Project Analyst,"[Excel, R, Communication]"
1,Seniour Analyst,"[Power BI, Excel, R, Communication]"
2,Seniour Growth Analyst,"[SQL, Python, Power BI, Tableau, R, VBA, Commu..."
3,Market Intelligence,"[Excel, Tableau, R, Communication]"
4,Data and Reporting Analyst,"[Power BI, R]"
5,Analyst – Visual Analytics,"[Excel, Tableau, R, Communication]"
6,Jr. (Content) Quality Assurance Analyst,[R]
7,Real-Time Analyst (Workforce Managment),"[Excel, R, Communication]"
8,Advanced Excel Data Analyst,"[Excel, R]"
9,MS Excel Data Analyst,"[Excel, R, Communication]"


In [10]:
df_raw.isna().sum()

Title             0
Company           0
Location          1
Skill             0
Category          0
skills_matched    0
dtype: int64

In [11]:
from collections import Counter

all_skills = [skill for row in df_raw["skills_matched"] for skill in row]
skill_counts = Counter(all_skills)
skill_counts.most_common()

[('R', 43),
 ('Communication', 26),
 ('Excel', 23),
 ('SQL', 16),
 ('Python', 12),
 ('Power BI', 11),
 ('Tableau', 9),
 ('Machine Learning', 6),
 ('Statistics', 6),
 ('Data Warehousing', 5),
 ('ETL', 5),
 ('Pandas', 4),
 ('Looker', 3),
 ('Spark', 3),
 ('Data Modeling', 3),
 ('Snowflake', 3),
 ('VBA', 1)]

In [12]:
import re

def extract_skills(text):
    if pd.isna(text):
        return []
    text = str(text).lower()
    matched = []
    for kw in SKILL_KEYWORDS:
        pattern = r'\b' + re.escape(kw.lower()) + r'\b'
        if re.search(pattern, text):
            matched.append(kw)
    return matched

df_raw["skills_matched"] = df_raw["Skill"].apply(extract_skills)

all_skills = [skill for row in df_raw["skills_matched"] for skill in row]
skill_counts = Counter(all_skills)
skill_counts.most_common()

[('Communication', 22),
 ('Excel', 15),
 ('SQL', 15),
 ('Python', 11),
 ('Tableau', 8),
 ('Power BI', 8),
 ('Machine Learning', 6),
 ('Statistics', 6),
 ('Data Warehousing', 5),
 ('ETL', 5),
 ('Pandas', 4),
 ('R', 3),
 ('Looker', 3),
 ('Spark', 3),
 ('Data Modeling', 3),
 ('Snowflake', 3)]

In [13]:
df_so = pd.read_csv("D:/PK-Data-Job-Analysis/data/raw/survey_results_public.csv") 
df_global = pd.read_csv("D:/PK-Data-Job-Analysis/data/raw/salaries.csv")

print("Shape of StackOverflow:", df_so.shape)
print(df_so.columns.tolist())
print("Shape of Global Salaries:", df_global.shape)
print(df_global.columns.tolist())   

C:\Users\Admin\AppData\Local\Temp\ipykernel_11544\1198103758.py:1: DtypeWarning: Columns (0: JobSatPoints_15_TEXT, 1: DatabaseHaveEntry, 2: DevEnvHaveEntry, 3: SOTagsHaveEntry, 4: SOTagsWant Entry, 5: OfficeStackWantEntry, 6: CommPlatformHaveEntr, 7: CommPlatformWantEntr, 8: SO_Actions_15_TEXT, 9: AIAgentOrchestration, 10: AIAgentObsWrite) have mixed types. Specify dtype option on import or set low_memory=False.
  df_so = pd.read_csv("D:/PK-Data-Job-Analysis/data/raw/survey_results_public.csv")


Shape of StackOverflow: (49191, 172)
['ResponseId', 'MainBranch', 'Age', 'EdLevel', 'Employment', 'EmploymentAddl', 'WorkExp', 'LearnCodeChoose', 'LearnCode', 'LearnCodeAI', 'AILearnHow', 'YearsCode', 'DevType', 'OrgSize', 'ICorPM', 'RemoteWork', 'PurchaseInfluence', 'TechEndorseIntro', 'TechEndorse_1', 'TechEndorse_2', 'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5', 'TechEndorse_6', 'TechEndorse_7', 'TechEndorse_8', 'TechEndorse_9', 'TechEndorse_13', 'TechEndorse_13_TEXT', 'TechOppose_1', 'TechOppose_2', 'TechOppose_3', 'TechOppose_5', 'TechOppose_7', 'TechOppose_9', 'TechOppose_11', 'TechOppose_13', 'TechOppose_16', 'TechOppose_15', 'TechOppose_15_TEXT', 'Industry', 'JobSatPoints_1', 'JobSatPoints_2', 'JobSatPoints_3', 'JobSatPoints_4', 'JobSatPoints_5', 'JobSatPoints_6', 'JobSatPoints_7', 'JobSatPoints_8', 'JobSatPoints_9', 'JobSatPoints_10', 'JobSatPoints_11', 'JobSatPoints_13', 'JobSatPoints_14', 'JobSatPoints_15', 'JobSatPoints_16', 'JobSatPoints_15_TEXT', 'AIThreat', 'NewRole

In [16]:
df_so.info()

<class 'pandas.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Columns: 172 entries, ResponseId to JobSat
dtypes: float64(52), int64(1), str(119)
memory usage: 176.2 MB


In [17]:
# Stack Overflow: keep only relevant columns, filter to data-related roles
so_cols = ["Country", "DevType", "RemoteWork", "LanguageHaveWorkedWith",
           "DatabaseHaveWorkedWith", "ConvertedCompYearly", "YearsCode"]
df_so_slim = df_so[so_cols].copy()

# Filter to data-related roles only (this is a text-contains filter since
# DevType often has combined values like "Data scientist or machine learning specialist")
data_roles_mask = df_so_slim["DevType"].str.contains(
    "Data scientist|Data analyst|Data engineer", case=False, na=False
)
df_so_data = df_so_slim[data_roles_mask].copy()
print("SO data-role responses:", df_so_data.shape)

# Global salaries: filter to data-related job titles the same way
global_mask = df_global["job_title"].str.contains(
    "Data Analyst|Data Scientist|Data Engineer", case=False, na=False
)
df_global_data = df_global[global_mask].copy()
print("Global data-role postings:", df_global_data.shape)

SO data-role responses: (1344, 7)
Global data-role postings: (49065, 11)


In [18]:
from collections import Counter

# LanguageHaveWorkedWith and DatabaseHaveWorkedWith are semicolon-separated lists
def split_and_count(series):
    all_items = []
    for val in series.dropna():
        all_items.extend([x.strip() for x in val.split(";")])
    return Counter(all_items)

lang_counts = split_and_count(df_so_data["LanguageHaveWorkedWith"])
db_counts = split_and_count(df_so_data["DatabaseHaveWorkedWith"])

print("Top languages (global data roles):")
print(lang_counts.most_common(10))
print("\nTop databases (global data roles):")
print(db_counts.most_common(10))

Top languages (global data roles):
[('Python', 913), ('SQL', 754), ('Bash/Shell (all shells)', 510), ('HTML/CSS', 394), ('JavaScript', 389), ('R', 241), ('PowerShell', 224), ('Java', 205), ('TypeScript', 159), ('C++', 146)]

Top databases (global data roles):
[('PostgreSQL', 524), ('Microsoft SQL Server', 336), ('MySQL', 304), ('SQLite', 298), ('BigQuery', 193), ('Databricks SQL', 187), ('MongoDB', 177), ('Snowflake', 172), ('Oracle', 142), ('DuckDB', 141)]


In [19]:
roi = df_global_data.groupby(["experience_level", "remote_ratio"])["salary_in_usd"].agg(
    ["median", "count"]
).reset_index()
print(roi)

   experience_level  remote_ratio    median  count
0                EN             0   83000.0   5558
1                EN            50   50000.0     62
2                EN           100   82000.0   1135
3                EX             0  175000.0    747
4                EX            50  130873.5      6
5                EX           100  220000.0    224
6                MI             0  118000.0  10834
7                MI            50   65019.0     74
8                MI           100  112400.0   2785
9                SE             0  150000.0  19852
10               SE            50   82382.0     44
11               SE           100  150000.0   7744
